# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema, available at:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Let's load metadata and records from the dataset using `mlcroissant`. This step loads both the Croissant schema and enables access to record sets, fields, and columns using their `@id` as required.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("=== Dataset Overview ===")
print("Name: ", metadata.name)
print("Identifier: ", metadata.identifier)
print("Description: ", metadata.description)
print("Authors (@id):")
for author in getattr(metadata, 'author', []):
    print("  -", author['@id'])
print("Record Sets (@id):", getattr(metadata, 'recordSet', []))
print("License: ", getattr(metadata, 'license', None))

## 2. Data Overview
In this section, we review available record sets, fields, and their IDs. All entities will be referenced strictly with their `@id`.

If the dataset has multiple record sets, we will enumerate them, together with their fields and columns.


In [ ]:
# List all available record sets and the columns (fields), referencing by @id
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found in metadata.")
else:
    print("=== Record Sets Overview ===")
    for idx, rs in enumerate(record_sets):
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
        print(f"Record Set {idx+1}: @id = {rs_id}")
        # Get all fields for the record set
        fields = dataset.record_set_fields(record_set=rs_id)
        print("  Fields (@id):")
        for field in fields:
            print(f"    - {field['@id']}")
        print()
        # Explore a sample record from this record set
        records_iter = dataset.records(record_set=rs_id)
        sample_record = None
        try:
            sample_record = next(records_iter)
        except StopIteration:
            pass
        if sample_record:
            print(f"  Sample record:")
            pprint.pprint(sample_record)
        print('-'*40)

## 3. Data Extraction
Let's load data from one or more record sets into Pandas DataFrames for analysis. For each record set, we'll use its `@id` and reference all fields by their `@id`. This makes downstream processing robust and schema-compliant.

If the dataset has no record sets, this step will be skipped; otherwise, DataFrames will be created.


In [ ]:
dataframes = {}
record_sets = getattr(metadata, 'recordSet', [])

if record_sets:
    print("=== Extracting DataFrames for Each Record Set ===")
    # Extract records for each record set using @id
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Record set @id: {rs_id}")
        print("Columns (@id):", df.columns.tolist())
        print(df.head(), "\n")
else:
    print("No record sets found. Skipping extraction.")

## 4. Exploratory Data Analysis (EDA)
To demonstrate processing, we'll select a record set and field (column) via their `@id`. We'll then filter, normalize, and group the data. If numeric fields are present, we'll operate on those.

Replace `<record_set_id>` and `<numeric_field_id>` below with relevant values from the above DataFrames.


In [ ]:
# Example EDA: Choose one record set with numeric fields, referenced by @id
if dataframes:
    # Get first record set id
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"EDA for record set @id: {record_set_id}")
    # Identify numeric columns (fields)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        print("No numeric fields found in first record set.")
    else:
        numeric_field = numeric_fields[0]  # Pick first numeric field by @id
        print(f"Using numeric field @id: {numeric_field}")
        threshold = df[numeric_field].mean() if not pd.isna(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where '{numeric_field}' > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() else 1)
        print(f"Normalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field if available
        possible_group_fields = [col for col in df.columns if col != numeric_field]
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped (mean) by field @id: {group_field}")
            print(grouped_df.head())
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field or explore relationships between two fields, referencing them always by their `@id`. For demonstration, a simple histogram and scatter (if two numeric fields are present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]    

    if numeric_fields:
        numeric_field = numeric_fields[0]
        plt.figure(figsize=(6, 4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f"Distribution of '{numeric_field}' (@id)")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()

        # If there are at least two numeric fields, show scatter plot
        if len(numeric_fields) >= 2:
            nf2 = numeric_fields[1]
            plt.figure(figsize=(6, 4))
            sns.scatterplot(x=df[numeric_field], y=df[nf2])
            plt.xlabel(numeric_field)
            plt.ylabel(nf2)
            plt.title(f"Scatter of '{numeric_field}' vs '{nf2}' (@id)")
            plt.show()
    else:
        print("No numeric fields for visualization.")
else:
    print("No dataframes available for visualization.")

## 6. Conclusion
This notebook has demonstrated loading, exploring, and processing a Croissant dataset using the `mlcroissant` library. Data entities were referenced strictly by their `@id`, ensuring schema-aligned usage. For real-world analysis, consult the dataset's field descriptions and carefully interpret clinical measurements, especially given cohort limitations (N=3, all female, single clinical site).

Use this workflow to apply FAIR best practices and reproducibility to small tabular biomedical datasets.